# Import libraries

In [30]:
import numpy as np
import pandas as pd
import scipy
import os
from scipy.optimize import minimize

# Import task data

In [31]:
data = pd.read_csv('../../task/data/1_data.csv')
choices = data['choice']
rewards = data['reward']
cue = data['condition']

# Define Models

## Basic Rescorla-Wagner model

In [32]:
def rescorla_wagner_basic(params, choices, rewards):
    alpha = params[0]  
    beta = params[1]  
    V = np.array([0.5, 0.5])# np.zeros(2)  # Initiale Values für beide Arme (=0)
    log_likelihood = 0

    for i in range(len(choices)):
        choice = choices[i]
        reward = rewards[i]
        #softmax
        prob_action = (np.exp(V[choice]*beta))/(np.exp(V[0]*beta)+np.exp(V[1]*beta))
       
        prediction_error = reward - V[choice]
        V[choice] += alpha * prediction_error
        # Negative Log-Likelihood (zu minimieren)
        log_likelihood += np.log(max(prob_action, 1e-10))  # Schutz vor log(0)

    return -log_likelihood

## Extended Rescorla-Wagner model
alpha_shift parameter added to differ learning rates for salient and non-salient feedback trials

In [33]:
def rescorla_wagner_extended(params, choices, rewards, cue):
    alpha = params[0]  # Lernrate ohne salient cue
    alpha_shift = params[1]  # Lernrate mit salient cue
    beta = params[2]
    V = np.array([0.5, 0.5])# np.zeros(2)  # Initiale Values für beide Arme (=0)
    log_likelihood = 0
    

    for i in range(len(choices)):
        choice = choices[i]
        reward = rewards[i]
        # Apply the softmax transformation
        prob_action = (np.exp(V[choice]*beta))/(np.exp(V[0]*beta)+np.exp(V[1]*beta))
        
        # Apply reinforcement learning model
        prediction_error = reward - V[choice]
        V[choice] = V[choice] + (alpha + alpha_shift * cue[i]) * prediction_error
        
        # Negative Log-Likelihood (zu minimieren)
        log_likelihood += np.log(max(prob_action, 1e-10))  # Schutz vor log(0)
        

    return -log_likelihood

# Model fitting and output

## Basic model

In [34]:
initial_params = [.5, 1]  # Initiale Schätzungen für die Lernraten
result = minimize(
    rescorla_wagner_basic, 
    initial_params, 
    args=(choices, rewards), 
    bounds=[(0, 1), (0, 10000)]
    )

alpha, beta = result.x

print(f"Optimierte Lernrate: {alpha}")
print(f"Softmax parameter: {beta}")


Optimierte Lernrate: 0.6087763829106462
Softmax parameter: 3.4736164523814606


## Extended model

In [35]:
initial_params = [.5, 0, 1]  # Initiale Schätzungen für die Lernraten
result = minimize(
    rescorla_wagner_extended, 
    initial_params, 
    args=(choices, rewards, cue), 
    bounds=[(0, 1), (-1, 1), (0, 10000)]
    )

alpha, alpha_shift, beta = result.x

print(f"Optimierte Lernrate: {alpha}")
print(f"Optimierte Shift-Lernrate: {alpha_shift}")
print(f"Softmax parameter: {beta}")

Optimierte Lernrate: 0.6124530540475184
Optimierte Shift-Lernrate: 0.5304214722670331
Softmax parameter: 3.213132802951154
